# Batched generation & padding side

Compare HuggingFace `generate` outputs across three strategies on the same
Qwen2.5-3B-Instruct model:

1. **Case 1** — sequential generation, one prompt at a time (reference).
2. **Case 2** — batched generation with **left** padding (correct for causal LMs).
3. **Case 3** — batched generation with **right** padding (incorrect; for contrast).

Then repeat the batched-vs-sequential comparison for prompt log-probabilities
and last-token embeddings to verify the padding/masking logic.

## Setup

In [ ]:
import gc

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Dataset and model paths
base_dir = '/groups/chichengz/tnn/datasets/'

dataset_dir = base_dir + "/prm800k/math_splits"

# Causal LM under test (swap to Llama-3.2-1B-Instruct to compare)
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
llm_dir = base_dir + "Qwen2.5-3B-Instruct"

# Process reward model (not used in this notebook)
prm_dir = base_dir + "Llama3.1-8B-PRM-Deepseek-Data"

In [ ]:
# Load tokenizer and causal LM onto GPU 0 in eval mode.
# torch_dtype="auto" picks up Qwen2.5's bf16 weights (vs. the fp32 default,
# which would double GPU memory for no accuracy gain).
device = "cuda:0"
tokenizer = AutoTokenizer.from_pretrained(llm_dir)
llm_tf = AutoModelForCausalLM.from_pretrained(
    llm_dir,
    torch_dtype="auto",
    device_map=device,
)
llm_tf.eval()

gc.collect()
torch.cuda.empty_cache()
print('#--- memory:', torch.cuda.memory_allocated(0) / (1024**3))

## Test prompts

In [ ]:
# Sample prompts of varying lengths to exercise padding behavior
texts = [
    "Hello, how are you?",
    "What is your name?",
    "Tell me a joke.",
    "Explain quantum computing in simple terms."
]

## Generation: padding strategies

For each strategy below, `generate` is called on the same prompts with the
same seed. Compare the outputs: Case 1 and Case 2 should agree; Case 3
should differ (and look worse) on the shorter prompts that get right-padded.

In [ ]:
# Case 1: Sequential generation (no batching, no padding).
# Reference behavior - each prompt is generated independently.
torch.manual_seed(100000 + 0)
torch.cuda.manual_seed(100000 + 0)

completions_no_padding = []
for text in texts:
    inputs = tokenizer(text, return_tensors="pt").to(llm_tf.device)
    with torch.no_grad():
        output_tokens = llm_tf.generate(
            **inputs,
            max_new_tokens=20,
            pad_token_id=tokenizer.pad_token_id,
        )
    # Strip the prompt so we display only the model's continuation
    prompt_len = inputs['input_ids'].shape[1]
    new_tokens = output_tokens[0, prompt_len:]
    completions_no_padding.append(tokenizer.decode(new_tokens, skip_special_tokens=True))

for i, (prompt, completion) in enumerate(zip(texts, completions_no_padding)):
    print(f"Input {i+1}:  {prompt}")
    print(f"Output {i+1}: {completion}")
    print()

In [ ]:
# Case 2: Batched generation with LEFT padding.
# Left padding is the correct choice for causal LM generation: real tokens stay
# right-aligned, so the model conditions on the last real token when emitting
# new tokens. Continuations should match Case 1 (modulo numerical drift).
torch.manual_seed(100000 + 0)
torch.cuda.manual_seed(100000 + 0)

# Set padding side explicitly - the tokenizer is shared state and any prior
# cell may have left it on the wrong side.
tokenizer.padding_side = 'left'
inputs = tokenizer(texts, return_tensors="pt", padding=True).to(llm_tf.device)

with torch.no_grad():
    output_tokens = llm_tf.generate(
        **inputs,
        max_new_tokens=20,
        pad_token_id=tokenizer.pad_token_id,
    )

# Strip the (padded) prompt to compare only the model's continuations
prompt_len = inputs['input_ids'].shape[1]
new_tokens = output_tokens[:, prompt_len:]
completions = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
assert len(completions) == len(texts)

for i, (prompt, completion) in enumerate(zip(texts, completions)):
    print(f"Input {i+1}:  {prompt}")
    print(f"Output {i+1}: {completion}")
    print()

In [ ]:
# Case 3: Batched generation with RIGHT padding.
# Right padding misaligns shorter prompts during generation - pads sit between
# the prompt and the new tokens, which typically degrades output quality.
# Included for comparison against Case 2.
torch.manual_seed(100000 + 0)
torch.cuda.manual_seed(100000 + 0)

# Set padding side explicitly - the tokenizer is shared state and any prior
# cell may have left it on the wrong side.
tokenizer.padding_side = 'right'
inputs = tokenizer(texts, return_tensors="pt", padding=True).to(llm_tf.device)

with torch.no_grad():
    output_tokens = llm_tf.generate(
        **inputs,
        max_new_tokens=20,
        pad_token_id=tokenizer.pad_token_id,
    )

# Strip the (padded) prompt - on right padding the pads sit between prompt and
# new tokens, so slicing by prompt_len excludes them from the displayed text.
prompt_len = inputs['input_ids'].shape[1]
new_tokens = output_tokens[:, prompt_len:]
completions = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
assert len(completions) == len(texts)

for i, (prompt, completion) in enumerate(zip(texts, completions)):
    print(f"Input {i+1}:  {prompt}")
    print(f"Output {i+1}: {completion}")
    print()

## Hidden states & log-probabilities

Compute prompt log-probability and the last-real-token hidden state for each
sequence — first as a single batched forward pass (with right padding +
attention-mask bookkeeping), then one sequence at a time as a reference.
Per-sequence totals should match between the two.

In [ ]:
# Batched forward pass: compute prompt log-probabilities and extract the
# last-token embedding for each sequence. Uses RIGHT padding so that
# position-0 of every sequence is a real token; the last-real-token index
# is recovered from attention_mask to skip the trailing pads.
torch.manual_seed(100000 + 0)
torch.cuda.manual_seed(100000 + 0)

# Set padding side explicitly - the tokenizer is shared state and any prior
# cell may have left it on the wrong side.
tokenizer.padding_side = 'right'
inputs = tokenizer(texts, return_tensors="pt", padding=True).to(llm_tf.device)

with torch.no_grad():
    outputs = llm_tf(**inputs, output_hidden_states=True)

    # hidden_states[-1]: last layer, shape (batch_size, seq_len, hidden_dim)
    hidden_states = outputs.hidden_states[-1]
    # logits: shape (batch_size, seq_len, vocab_size)
    logits = outputs.logits

# Shift for LM scoring: logit at position i predicts token at position i+1
shifted_labels = inputs['input_ids'][:, 1:]
shifted_logits = logits[:, :-1, :]

# Gather log-probability of the actual next token at each position
log_probs = F.log_softmax(shifted_logits, dim=-1)
selected_log_probs = log_probs.gather(2, shifted_labels.unsqueeze(-1)).squeeze(-1)
print(selected_log_probs.shape)

# Zero out contributions from padding positions before summing
attention_mask = inputs['attention_mask'][:, 1:]
selected_log_probs = selected_log_probs * attention_mask
total_log_probs = selected_log_probs.sum(dim=1)

# Last non-pad token index per sequence -> grab its hidden state
last_token_indices = inputs['attention_mask'].sum(dim=1) - 1
last_token_embeddings = hidden_states[torch.arange(hidden_states.size(0)), last_token_indices]

for i, text in enumerate(texts):
    print(f"\n=== Input {i+1}: {text}")
    print(f"Last Token Embedding (5 dims): {last_token_embeddings[i][:5]}")
    print(f"Total Log Probability         : {total_log_probs[i].item()}")

In [ ]:
# Reference: compute the same quantities one sequence at a time (no padding).
# Per-sequence totals should match the batched version above; if they don't,
# the batched padding/masking logic is wrong.
torch.manual_seed(100000 + 0)
torch.cuda.manual_seed(100000 + 0)

for text in texts:
    inputs = tokenizer(text, return_tensors="pt").to(llm_tf.device)
    with torch.no_grad():
        outputs = llm_tf(**inputs, output_hidden_states=True)
        logits = outputs.logits                    # (1, seq_len, vocab_size)
        hidden_states = outputs.hidden_states[-1]  # (1, seq_len, hidden_dim)

    # Shift for LM scoring (same as the batched version)
    shifted_logits = logits[:, :-1, :]
    shifted_labels = inputs['input_ids'][:, 1:]

    log_probs = F.log_softmax(shifted_logits, dim=-1)
    selected_log_probs = log_probs.gather(2, shifted_labels.unsqueeze(-1)).squeeze(-1)
    total_log_prob = selected_log_probs.sum()

    # Without padding, "last token" is simply the final position
    last_token_embedding = hidden_states[0, -1]

    print(f"\n=== Input: {text}")
    print(f"Last Token Embedding (5 dims): {last_token_embedding[:5]}")
    print(f"Total Log Probability         : {total_log_prob.item()}")